# 🌿 Darukaa Reference Benchmarking Pipeline — v0.2.0

Profile-first, non-compensatory, evidence-graded biodiversity benchmarking.
This notebook runs the pipeline and displays its outputs; it does **not** recompute a
composite (the pipeline does that internally, correctly). See `METHODOLOGY_MASTER.md`
and `ASSUMPTIONS_AND_LIMITATIONS.md` before quoting any result.

## 1. Setup — clone repo & install

In [ ]:
# Clone (or update) the repo and install it.
import os
REPO = 'reference-benchmarking'
if not os.path.exists(REPO):
    !git clone https://github.com/G-auravSingh/reference-benchmarking.git
%cd $REPO/darukaa_reference_v0.2.1
!pip install -q -r requirements.txt
!pip install -q -e .

## 2. Authenticate Google Earth Engine
Uses your GEE project (e.g. `gaurav-singh-007`).

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='gaurav-singh-007')  # <- your GEE project id
print('Earth Engine ready')

## 3. Upload your site KML/KMZ
One or more polygons (a single site, or DBSCAN assessment-cluster tiles from the
site-selection pipeline for large agroforestry AOIs).

In [ ]:
from google.colab import files
up = files.upload()
SITE_PATH = list(up.keys())[0]
print('Using:', SITE_PATH)

## 4. Configure the run
Everything is driven by one `Config`. The important v0.2.0 knobs are shown; all have
safe defaults. `reference_stratification` defaults to `ecoregion_landcover` (the
SEED-faithful path) — **verify the PNV crosswalk before trusting a live run** (see
README §6 and `ASSUMPTIONS_AND_LIMITATIONS.md`).

In [ ]:
from darukaa_reference.config import Config

config = Config(
    gee_project='gaurav-singh-007',
    output_dir='outputs',
    output_format='both',              # json + csv (+ html always)
    # --- project context (shared with site-selection) ---
    realm='terrestrial', archetype='conservation', assessment_mode='baseline',
    # --- reference (SEED) ---
    hmi_hard_ceiling=0.05,             # SEED maximum
    use_variance_stability_floor=True, # OD-3: suppress score on a noisy reference
    reference_stratification='ecoregion_landcover',  # default, SEED-faithful; see README §6
    # --- optional SEED kernel view (OD-4) ---
    use_seed_kernel=False, seed_kernel_delta=0.5,
)
print('archetype=%s mode=%s stratification=%s' % (config.archetype, config.assessment_mode, config.reference_stratification))

## 5. Run the pipeline
Loads sites → resolves ecoregions → builds SEED references → benchmarks each indicator
→ profile-first non-compensatory scoring → writes `outputs/benchmark_scorecard.json/.csv/.html`.
Removed/inactive indicators (e.g. CERI) are skipped automatically.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
from darukaa_reference.pipeline import Pipeline

report = Pipeline(config).run(site_path=SITE_PATH)
print('\nStatus:', {k: len(v) for k, v in report['indicator_status'].items()})

## 6. What is scored, and what is not
Every indicator's honest disposition — nothing hidden.

In [ ]:
import pandas as pd
st = report['indicator_status']
for k in ['scored','contextual','screening_only','pending_inputs','removed']:
    print(f"{k:16s} ({len(st.get(k,[]))}): {', '.join(st.get(k,[])) or '—'}")

## 7. Site profiles (profile-first)
The primary output: per-component limiting factor, the condition × pressure decision,
and the secondary roll-up with its stability flag. Read the minimum component first.

In [ ]:
for site_id, prof in report.get('site_profiles', {}).items():
    cond = prof.get('condition', {}); sens = cond.get('sensitivity', {})
    print(f"\n=== {site_id} — decision: {prof.get('matrix_cell')} ===")
    for comp, cs in prof.get('components', {}).items():
        print(f"  {comp:16s} {cs['headline']:.3f}  (limiting: {cs['limiting_subdimension']})")
    if cond.get('rollup') is not None:
        stab = 'stable' if sens.get('stable') else 'UNSTABLE'
        print(f"  roll-up {cond['rollup']:.3f} | min {cond['minimum']:.3f} @ {cond['minimum_component']} | sensitivity {stab}")
    if prof.get('pressure', {}).get('headline') is not None:
        print(f"  pressure {prof['pressure']['headline']:.3f}")

## 8. Indicator scorecard

In [ ]:
df = pd.DataFrame(report['scorecard'])
cols = [c for c in ['site_id','indicator','construct','evidence_tier',
        'site_value','tier2_benchmark','tier2_benchmark_estimator',
        'tier2_display_pct_of_reference','reference_type'] if c in df.columns]
df[cols]

## 9. Evidence-graded HTML report
The client deliverable — a deterministic projection of the registry, generated every run.

In [ ]:
from IPython.display import HTML, FileLink
html_path = 'outputs/benchmark_scorecard.html'
display(FileLink(html_path))
HTML(open(html_path).read())

## 10. Download results

In [ ]:
from google.colab import files
for ext in ['json','csv','html']:
    p = f'outputs/benchmark_scorecard.{ext}'
    if os.path.exists(p): files.download(p)

---
## 11. Multi-tile / agroforestry projects (alternative to steps 3-10 above)

Use this section instead of steps 3-10 when your project has many scattered
parcels (agroforestry) rather than one compact site (conservation). Each **tile**
may itself contain many individual farm parcels — they are dissolved into one
geometry per tile automatically. The project-level result is combined
**non-compensatorily**: the project's signal for every indicator is set by its
**worst tile**, named explicitly — never averaged away by a larger, better tile.

**You need the tiles already produced** (e.g. via DBSCAN grouping of parcel
centroids, done upstream — this notebook does not do the tiling itself). If you
only have one KML with hundreds of scattered parcels and no tiles yet, stop here
and produce tiles first.

In [ ]:
# Upload ALL tile files for this project (one KML/GeoJSON per tile).
# You can select multiple files at once in the upload dialog.
from google.colab import files
up = files.upload()
TILE_PATHS = list(up.keys())
TILE_LABELS = [p.rsplit('.', 1)[0] for p in TILE_PATHS]  # filename (no ext) as label
print(f'{len(TILE_PATHS)} tiles uploaded:')
for p, l in zip(TILE_PATHS, TILE_LABELS):
    print(f'  {l}  <-  {p}')

In [ ]:
# Same Config as the single-site flow — just set archetype='agroforestry'.
# If you already ran Section 4 above for a single-site run, you can reuse
# that `config` object instead of rebuilding it here.
from darukaa_reference.config import Config
from darukaa_reference.indicators import create_default_registry

config = Config(
    gee_project='gaurav-singh-007',
    output_dir='outputs',
    archetype='agroforestry', realm='terrestrial', assessment_mode='baseline',
    hmi_hard_ceiling=0.05, use_variance_stability_floor=True,
    reference_stratification='ecoregion_landcover',
)
registry = create_default_registry()

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
from darukaa_reference.project_aggregation import run_multi_tile_project

PROJECT_NAME = 'my_agroforestry_project'  # <- CHANGE THIS

project = run_multi_tile_project(
    config, registry,
    tile_paths=TILE_PATHS, tile_labels=TILE_LABELS,
    project_name=PROJECT_NAME,
)
print('\nStatus:', {k: len(v) for k, v in project['indicator_status'].items()})
print('Tiles: %d succeeded, %d failed. Total area: %.1f ha' % (
    project['meta']['n_tiles_succeeded'], project['meta']['n_tiles_failed'],
    project['meta']['total_area_ha']))
if project['meta']['failed_tiles']:
    print('FAILED TILES:', project['meta']['failed_tiles'])

### Why the project profile looks like this — per-indicator worst tile

In [ ]:
import pandas as pd
rows = []
for name, s in project['multi_tile_summary']['per_indicator'].items():
    if s.get('status') == 'ok':
        rows.append({'indicator': name, 'worst_tile': s['worst_tile'],
                     'worst_tile_value': s['worst_tile_benchmark'],
                     'area_wtd_geomean_context': s['area_weighted_geomean_normalised'],
                     'tiles_with_data': f"{s['n_tiles_with_data']}/{s['n_tiles_total']}"})
pd.DataFrame(rows)

### Project-level profile (same profile-first output as a single site)

In [ ]:
prof = project['site_profiles']['PROJECT']
print('Decision:', prof['matrix_cell'])
for comp, cs in prof.get('components', {}).items():
    print(f"  {comp:16s} {cs['headline']:.3f}  (limiting: {cs['limiting_subdimension']})")
cond = prof.get('condition', {})
if cond.get('rollup') is not None:
    stab = 'stable' if cond.get('sensitivity',{}).get('stable') else 'UNSTABLE'
    print(f"  roll-up {cond['rollup']:.3f}  (min {cond['minimum']:.3f} @ {cond['minimum_component']}; sensitivity {stab})")

### Project-level evidence-graded HTML report

In [ ]:
from IPython.display import HTML, FileLink
html_path = f'outputs/{PROJECT_NAME}_project.html'
display(FileLink(html_path))
HTML(open(html_path).read())

### Download everything (project-level + every individual tile's own report)

In [ ]:
from google.colab import files
import shutil, os

# Zip the whole outputs folder so project-level AND per-tile files come as one download.
shutil.make_archive(f'{PROJECT_NAME}_results', 'zip', 'outputs')
files.download(f'{PROJECT_NAME}_results.zip')

---
### Monitoring mode (cycle 2+)
Set `assessment_mode='monitoring'`, run this cycle, then diff against the stored Year-0
report with `change.py`:


In [ ]:
# from darukaa_reference import change as CH
# import json
# base = CH.from_report(json.load(open('outputs/benchmark_scorecard_YEAR0.json')))
# cur  = CH.from_report(report)
# res  = CH.score_cycle(base, cur)
# print(res['summary'])  # per-indicator change + BACI where controls exist

### Adding a custom indicator
Register with the full contract so eligibility is computed correctly (see `INDICATOR_REGISTER.md`).

In [ ]:
# from darukaa_reference.registry import IndicatorRegistry
# r = create_default_registry()
# r.register(name='my_metric', display_name='My Metric', source_type='gee',
#            extract_fn=my_fn, construct='C2_vegetation', subdimension='structure',
#            measurement_scale='ratio', evidence_tier='baseline',
#            reference_type='contemporary_best_on_offer', uncertainty_method='bootstrap_ci',
#            input_layers=['my_layer'])